# 真实集成：长回答的自一致性与证据支持

API 真实生成/采样，本地 NLI 同时服务于 SelfCheck 和 FActScore，但两个信号独立返回。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai,nli]"`；需要 API key 和本地 NLI 模型。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `OPENAI_MODEL: str`：真实生成模型。
- `LOCAL_NLI_MODEL: str`：本地 NLI 模型。
- `NUM_SAMPLES: int = 5`：黑盒采样数。
- `claims: list[str] | None`：可选显式声明。

## 步骤 1：配置真实后端

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
from openai import OpenAI
from hallucination.detection.backends.api import OpenAIResponsesBackend
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.8
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
base_url = os.environ.get("OPENAI_BASE_URL")
client = OpenAI(api_key=api_key, **({"base_url": base_url} if base_url else {}))
backend = OpenAIResponsesBackend(OPENAI_MODEL, client=client, temperature=TEMPERATURE)

from hallucination.detection.backends.nli import BidirectionalNLIBackend
LOCAL_NLI_MODEL = "/absolute/path/to/your/nli-model"
NLI_DEVICE = "cpu"
if not Path(LOCAL_NLI_MODEL).is_dir():
    raise ValueError("Set LOCAL_NLI_MODEL to an existing local NLI model directory.")
nli = BidirectionalNLIBackend(LOCAL_NLI_MODEL, device=NLI_DEVICE)

## 步骤 2：建立分离 pipeline

In [ ]:
from hallucination.detection.blackbox.selfcheck_nli import SelfCheckNLIScorer
from hallucination.detection.evidence import FActScoreScorer, LocalCorpusRetriever, NLIEvidenceVerifier, load_textbook_evidence
from hallucination.detection.pipelines import EvidenceVerificationPipeline, LongFormVerificationPipeline
from hallucination.detection.pipelines.short_answer import BlackBoxSamplingPipeline
sampling = BlackBoxSamplingPipeline(backend, sentence_scorers=[SelfCheckNLIScorer(nli)], num_samples=5)
evidence = EvidenceVerificationPipeline(FActScoreScorer(NLIEvidenceVerifier(nli), retriever=LocalCorpusRetriever(load_textbook_evidence())))
workflow = LongFormVerificationPipeline(sampling, evidence_pipeline=evidence)

## 步骤 3：运行并分别读取

In [ ]:
prompt = input("Prompt：").strip()
claims_text = input("可选 claims，用 | 分隔；直接回车则按句切分：").strip()
claims = [item.strip() for item in claims_text.split("|") if item.strip()] or None
result = workflow.detect_prompt(prompt, claims=claims)
print("response:", result.self_consistency.metadata["response"])
print("self consistency:", [(unit.text, unit.risk) for unit in result.self_consistency.units])
print("evidence support:", [(claim.claim, claim.label) for claim in result.evidence_support.claims])

## 结果解读与限制

稳定采样不能证明真实性，有限语料也不能证明完整事实。界面和决策层必须分别呈现两类结果。